<a href="https://colab.research.google.com/github/sharniks/langchain-fundamentals/blob/main/currency_convertor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') # Make sure you've added your HF_TOKEN to Colab secrets!

In [85]:
!pip install -q langchain-huggingface langchain-core requests

In [86]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [100]:
# tool create
@tool
def get_conversion_factor(base_currency: str, target_currency:str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  # Retrieve API key from Colab secrets
  EXCHANGE_RATE_API_KEY = userdata.get('EXCHANGE_RATE_API_KEY')
  url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

In [101]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784419201,
 'time_last_update_utc': 'Sun, 19 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784505601,
 'time_next_update_utc': 'Mon, 20 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.3793}

In [89]:
convert.invoke({'base_currency_value':10, 'conversion_rate':96.3793})

963.793

In [90]:
# tool binding
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-72B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [91]:
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [92]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [93]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [94]:
ai_message = llm_with_tools.invoke(messages)

In [95]:
messages.append(ai_message)

In [96]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_8JRwj9dzkI2RhLjhVmrdizbW',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_OrFVYl3pWHIUDo417dDL7ji9',
  'type': 'tool_call'}]

In [97]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to the message list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current argument
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    # append this tool message to the message list
    messages.append(tool_message2)

In [98]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_8JRwj9dzkI2RhLjhVmrdizbW', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_OrFVYl3pWHIUDo417dDL7ji9', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 446, 'total_tokens': 498}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f7bee-05f9-7ff1-8b5e-d5ded6e972a5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_8JRwj9dzkI2RhLjhVmrdi

In [99]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 96.3793. Based on this, 10 USD is approximately 963.793 INR.'